In [43]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device name:   {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
from google.colab import drive
drive.mount('/content/drive')

GPU available: True
Device name:   Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [44]:
!pip install transformers accelerate --quiet

In [45]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

file_id = '1hHBLa4O6-h6YQuBK3lZ7iiXaaFa9yM3J'
url = f'https://drive.google.com/uc?id={file_id}'

df = pd.read_csv(url)

# Combine title + description for BERT input
# BERT handles its own tokenization — no need for manual cleaning
df['bert_text'] = (
    "Title: " + df['title'].fillna('') + " [SEP] " +
    "Profile: " + df['company_profile'].fillna('') + " [SEP] " +
    "Requirements: " + df['requirements'].fillna('') + " [SEP] " +
    "Description: " + df['description'].fillna('')
)


X = df['bert_text'].values
y = df['fraudulent'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Fake in train: {y_train.sum()} ({y_train.mean()*100:.1f}%)")

Train: 14304 | Test: 3576
Fake in train: 693 (4.8%)


In [46]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DataCollatorWithPadding
from sklearn.utils.class_weight import compute_class_weight

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

class JobDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        encodings = tokenizer(
            list(texts),
            max_length=max_length,
            truncation=True,
            padding=False,
            return_tensors=None
        )
        self.input_ids = [torch.tensor(ids, dtype=torch.long) for ids in encodings['input_ids']]
        self.attention_mask = [torch.tensor(mask, dtype=torch.long) for mask in encodings['attention_mask']]
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }


# Initialize datasets
train_dataset = JobDataset(X_train, y_train, tokenizer)
test_dataset = JobDataset(X_test, y_test, tokenizer)

# Handle class imbalance
class_weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weights = np.sqrt(class_weights)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print("Class weights:", class_weights)

# Dynamic padding collator — pads each batch to its own longest sequence,
collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)

# DataLoaders
train_loader = DataLoader(
    train_dataset, batch_size=16, shuffle=True,
    num_workers=2, collate_fn=collator
)
test_loader = DataLoader(
    test_dataset, batch_size=32, shuffle=False,
    num_workers=2, collate_fn=collator
)

print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")

Class weights: [0.72488437 3.21252958]
Train batches: 894 | Test batches: 112


In [47]:
from transformers import DistilBertForSequenceClassification
from torch.amp import GradScaler, autocast
scaler = GradScaler('cuda')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

torch.backends.cudnn.benchmark = True

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
).to(device)

# Move class weights to device
class_weights_tensor = class_weights_tensor.to(device)

scaler = GradScaler()

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Training on: cuda


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model parameters: 66,955,010


In [48]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

EPOCHS       = 4
LR           = 1e-5
WARMUP_STEPS = 150
TOTAL_STEPS  = len(train_loader) * EPOCHS

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.1)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TOTAL_STEPS
)

loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)

In [49]:
from sklearn.metrics import f1_score, classification_report
from torch.amp import autocast
import numpy as np

def train_epoch(model, loader, optimizer, scheduler, loss_fn, scaler, device):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()

        with autocast('cuda'):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss    = loss_fn(outputs.logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        preds       = torch.argmax(outputs.logits, dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    return total_loss / len(loader), correct / total


def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0
    all_probs, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            with autocast('cuda'):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss    = loss_fn(outputs.logits, labels)

            total_loss += loss.item()
            probs = torch.softmax(outputs.logits, dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    best_thresh = 0.5
    best_f1 = 0
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    for thresh in np.arange(0.1, 0.9, 0.05):
        preds = (all_probs >= thresh).astype(int)
        score = f1_score(all_labels, preds)
        if score > best_f1:
            best_f1 = score
            best_thresh = thresh

    return total_loss / len(loader), best_f1, best_thresh, all_probs, all_labels


# Run training
print("Starting training...\n")
best_f1 = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, scheduler, loss_fn, scaler, device)
    val_loss, val_f1, opt_thresh, _, _ = evaluate(
        model, test_loader, loss_fn, device)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f}   | Val F1:    {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        model.save_pretrained('/content/drive/MyDrive/fakejob/models/distilbert')
        tokenizer.save_pretrained('/content/drive/MyDrive/fakejob/models/distilbert')
        print(f"  Best model saved (F1={best_f1:.4f})")
    print()

print(f"\nTraining complete. Best F1: {best_f1:.4f}")

Starting training...

Epoch 1/4
  Train Loss: 0.3309 | Train Acc: 96.32%
  Val Loss:   0.1770   | Val F1:    0.7815


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Best model saved (F1=0.7815)

Epoch 2/4
  Train Loss: 0.1450 | Train Acc: 98.36%
  Val Loss:   0.1417   | Val F1:    0.8754


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Best model saved (F1=0.8754)

Epoch 3/4
  Train Loss: 0.0815 | Train Acc: 99.22%
  Val Loss:   0.1558   | Val F1:    0.8949


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Best model saved (F1=0.8949)

Epoch 4/4
  Train Loss: 0.0504 | Train Acc: 99.54%
  Val Loss:   0.1601   | Val F1:    0.9052


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Best model saved (F1=0.9052)


Training complete. Best F1: 0.9052


In [52]:
import numpy as np
from sklearn.metrics import classification_report

best_model = DistilBertForSequenceClassification.from_pretrained(
    '/content/drive/MyDrive/fakejob/models/distilbert'
).to(device)
val_loss, final_f1, best_thresh, final_probs, final_labels = evaluate(
    best_model, test_loader, loss_fn, device
)

final_preds = (np.array(final_probs) >= best_thresh).astype(int)

print("=" * 50)
print(f"FINAL CLASSIFICATION REPORT (DistilBERT at Threshold {best_thresh:.2f})")
print("=" * 50)
print(classification_report(final_labels, final_preds, target_names=['Real', 'Fake']))


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

FINAL CLASSIFICATION REPORT (DistilBERT at Threshold 0.55)
              precision    recall  f1-score   support

        Real       0.99      1.00      1.00      3403
        Fake       0.96      0.86      0.91       173

    accuracy                           0.99      3576
   macro avg       0.98      0.93      0.95      3576
weighted avg       0.99      0.99      0.99      3576



In [54]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

In [57]:
import numpy as np
!pip install huggingface_hub --quiet
from huggingface_hub import notebook_login, HfApi
notebook_login()

HF_USERNAME = "Yuvraj-Bansal"
REPO_NAME   = f"{HF_USERNAME}/fake_job-distilbert"

best_model.push_to_hub(REPO_NAME)
tokenizer.push_to_hub(REPO_NAME)

print("Uploading optimized threshold file...")

threshold_path = '/content/drive/MyDrive/fakejob/models/best_threshold.txt'
np.savetxt(threshold_path, [best_thresh], fmt='%f')

api = HfApi()
api.upload_file(
    path_or_fileobj=threshold_path,
    path_in_repo='best_threshold.txt',
    repo_id=REPO_NAME,
    repo_type="model"
)

print(f"\n Model and threshold are live at: https://huggingface.co/{REPO_NAME}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...yvdqbq2/model.safetensors: 100%|##########|  268MB /  268MB            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Uploading optimized threshold file...


No files have been modified since last commit. Skipping to prevent empty commit.



 Model and threshold are live at: https://huggingface.co/Yuvraj-Bansal/fake_job-distilbert
